# Problemas variados de programación dinámica

La programación dinámica resuelve un problema a partir de subproblemas más pequeños que se repiten, calculando cada uno una sola vez. Se puede escribir **de arriba abajo**, con la recurrencia tal cual en una función recursiva memoizada con `functools.lru_cache`, o **de abajo arriba**, rellenando una tabla en un orden en el que cada casilla solo depende de casillas ya calculadas (lo que suele permitir guardar solo las últimas filas). En los dos casos el coste es

$$O(\text{número de estados} \times \text{transiciones por estado})$$

frente al coste exponencial de probar todas las posibilidades, que aquí sirve para comprobar los resultados en casos pequeños. Ideas de los ejercicios de [José A. Alonso](https://github.com/jaalonso/Ejercicios-Python) y [Edabit](https://edabit.com/challenges/python3).

## 1. Escalera con coste por escalón

Subir la escalera de un mirador cansa: pisar el escalón $i$ cuesta $c_i$ (según lo alto o lo resbaladizo que sea). Desde el suelo se sube de uno en uno o de dos en dos, se paga cada escalón que se pisa y se termina al llegar arriba, por encima del último. Si $M_i$ es el coste mínimo para quedar en la posición $i$ (con el suelo en la 0 y arriba en la $n + 1$, que no cuesta nada), $M_0 = 0$, $M_1 = c_1$ y $M_i = c_i + \min(M_{i-1}, M_{i-2})$. Calcular el coste mínimo de arriba abajo y de abajo arriba, devolver además qué escalones pisar y comprobarlo contra todas las formas de subir.

In [1]:
import itertools
import math
import random
from collections import Counter
from fractions import Fraction
from functools import lru_cache


def coste_minimo_memo(costes):
    c = [0, *costes, 0]

    @lru_cache(maxsize=None)
    def M(i):
        return c[i] if i <= 1 else c[i] + min(M(i - 1), M(i - 2))

    return M(len(c) - 1)


def coste_minimo(costes):
    c = [0, *costes, 0]
    M, anterior = [0] * len(c), [0] * len(c)
    M[1] = c[1]
    for i in range(2, len(c)):
        anterior[i] = i - 1 if M[i - 1] <= M[i - 2] else i - 2
        M[i] = c[i] + M[anterior[i]]
    pisados, i = [], anterior[-1]
    while i:
        pisados.append(i)
        i = anterior[i]
    return M[-1], pisados[::-1]


def caminos_hasta(q):
    """Listas de posiciones pisadas para ir de 0 a q con pasos de 1 o 2."""
    if q == 0:
        return [[]]
    return [camino + [q] for paso in (1, 2) if q - paso >= 0 for camino in caminos_hasta(q - paso)]


def coste_minimo_fuerza_bruta(costes):
    return min(sum(costes[i - 1] for i in camino[:-1]) for camino in caminos_hasta(len(costes) + 1))


escalera = [3, 8, 2, 9, 1, 7, 7, 2, 6, 4, 1, 9]
coste, pisados = coste_minimo(escalera)
print(f"Coste mínimo {coste} pisando los escalones {pisados}")
assert coste == coste_minimo_memo(escalera) == sum(escalera[i - 1] for i in pisados)

random.seed(1)
for _ in range(300):
    costes = [random.randint(0, 9) for _ in range(random.randint(0, 14))]
    coste, pisados = coste_minimo(costes)
    assert coste == coste_minimo_memo(costes) == coste_minimo_fuerza_bruta(costes)
    assert all(1 <= b - a <= 2 for a, b in itertools.pairwise([0, *pisados, len(costes) + 1]))

Coste mínimo 20 pisando los escalones [1, 3, 5, 7, 8, 10, 11]


## 2. Formas de subir con pasos dados

Contar de cuántas formas se puede subir una escalera de $n$ escalones si cada paso sube un número de escalones de un conjunto dado (por ejemplo, $\{1, 3, 5\}$); dos formas son distintas si la secuencia de pasos es distinta. El último paso fue de $p$ escalones para algún $p$ del conjunto, así que $W_0 = 1$ y $W_n = \sum_{p \le n} W_{n-p}$, en $O(n \cdot |P|)$. Escribir las versiones de arriba abajo y de abajo arriba, comprobar que con pasos $\{1, 2\}$ salen los números de Fibonacci y compararlo contra la generación de todas las secuencias.

In [2]:
def formas_memo(n, pasos):
    @lru_cache(maxsize=None)
    def W(m):
        return 1 if m == 0 else sum(W(m - p) for p in pasos if p <= m)

    return W(n)


def formas(n, pasos):
    W = [1] + [0] * n
    for m in range(1, n + 1):
        W[m] = sum(W[m - p] for p in pasos if p <= m)
    return W[n]


def secuencias(n, pasos):
    if n == 0:
        yield ()
    for p in pasos:
        if p <= n:
            for resto in secuencias(n - p, pasos):
                yield (p, *resto)


conjuntos = [(1, 2), (1, 3, 5), (2, 3), (1, 2, 3)]
print(f"{'n':>3}" + "".join(f"{str(set(c)):>12}" for c in conjuntos))
for n in range(13):
    print(f"{n:>3}" + "".join(f"{formas(n, c):>12}" for c in conjuntos))
    for c in conjuntos:
        assert formas(n, c) == formas_memo(n, c) == len(list(secuencias(n, c)))

fibonacci = [0, 1]
while len(fibonacci) < 92:
    fibonacci.append(fibonacci[-1] + fibonacci[-2])
assert all(formas(n, (1, 2)) == fibonacci[n + 1] for n in range(90))
print(f"\n100 escalones con pasos de 1, 2 o 3: {formas(100, (1, 2, 3))} formas")
print("Formas de subir 6 escalones con pasos de 1, 3 o 5:", *secuencias(6, (1, 3, 5)))

  n      {1, 2}   {1, 3, 5}      {2, 3}   {1, 2, 3}
  0           1           1           1           1
  1           1           1           0           1
  2           2           1           1           2
  3           3           2           1           4
  4           5           3           1           7
  5           8           5           2          13
  6          13           8           2          24
  7          21          12           3          44
  8          34          19           4          81
  9          55          30           5         149
 10          89          47           7         274
 11         144          74           9         504
 12         233         116          12         927

100 escalones con pasos de 1, 2 o 3: 180396380815100901214157639 formas
Formas de subir 6 escalones con pasos de 1, 3 o 5: (1, 1, 1, 1, 1, 1) (1, 1, 1, 3) (1, 1, 3, 1) (1, 3, 1, 1) (1, 5) (3, 1, 1, 1) (3, 3) (5, 1)


## 3. Particiones de un número

Una partición de $n$ es una forma de escribirlo como suma de enteros positivos sin importar el orden: $4 = 3 + 1 = 2 + 2 = 2 + 1 + 1 = 1 + 1 + 1 + 1$, así que $p(4) = 5$. Si $P(n, k)$ cuenta las particiones de $n$ con partes de tamaño $k$ como mucho, o no se usa ninguna parte $k$ o se quita una: $P(n, k) = P(n, k - 1) + P(n - k, k)$. Contarlas de arriba abajo y de abajo arriba ($O(n^2)$), generarlas con *backtracking* (partes de mayor a menor) y comprobar los recuentos con la generación, con los valores conocidos $1, 2, 3, 5, 7, 11, 15, 22, 30, 42$ para $n = 1 \dots 10$, $p(100) = 190\,569\,292$ y $p(200) = 3\,972\,999\,029\,388$, y con la recurrencia de los números pentagonales de Euler.

In [3]:
def particiones_memo(n):
    @lru_cache(maxsize=None)
    def P(m, k):
        if m == 0:
            return 1
        if m < 0 or k == 0:
            return 0
        return P(m, k - 1) + P(m - k, k)

    return P(n, n)


def particiones(n):
    p = [1] + [0] * n
    for parte in range(1, n + 1):
        for m in range(parte, n + 1):
            p[m] += p[m - parte]
    return p[n]


def genera_particiones(n, maxima=None):
    maxima = n if maxima is None else maxima
    if n == 0:
        yield ()
        return
    for parte in range(min(n, maxima), 0, -1):
        for resto in genera_particiones(n - parte, parte):
            yield (parte, *resto)


def particiones_euler(n):
    p = [1] + [0] * n
    for m in range(1, n + 1):
        for k in itertools.count(1):
            g1, g2 = k * (3 * k - 1) // 2, k * (3 * k + 1) // 2
            if g1 > m:
                break
            signo = 1 if k % 2 else -1
            p[m] += signo * (p[m - g1] + (p[m - g2] if g2 <= m else 0))
    return p[n]


print("Particiones de 6:", *(" + ".join(map(str, p)) for p in genera_particiones(6)), sep="\n  ")
assert [particiones(n) for n in range(1, 11)] == [1, 2, 3, 5, 7, 11, 15, 22, 30, 42]
for n in range(30):
    assert particiones(n) == particiones_memo(n) == len(list(genera_particiones(n))) == particiones_euler(n)
for n in range(30, 300, 17):
    assert particiones(n) == particiones_euler(n)
assert particiones(100) == particiones_memo(100) == 190_569_292
assert particiones(200) == 3_972_999_029_388
print(f"\np(100) = {particiones(100):,}   p(200) = {particiones(200):,}   p(1000) = {particiones(1000):.3e}")

Particiones de 6:
  6
  5 + 1
  4 + 2
  4 + 1 + 1
  3 + 3
  3 + 2 + 1
  3 + 1 + 1 + 1
  2 + 2 + 2
  2 + 2 + 1 + 1
  2 + 1 + 1 + 1 + 1
  1 + 1 + 1 + 1 + 1 + 1

p(100) = 190,569,292   p(200) = 3,972,999,029,388   p(1000) = 2.406e+31


## 4. Partes distintas y partes impares

Euler demostró que un número tiene tantas particiones en partes distintas como en partes impares: las de 6 en partes distintas son $6$, $5 + 1$, $4 + 2$ y $3 + 2 + 1$, y en partes impares, $5 + 1$, $3 + 3$, $3 + 1 + 1 + 1$ y $1 + 1 + 1 + 1 + 1 + 1$. Contar las primeras como una mochila 0/1 (cada parte se usa una vez como mucho, recorriendo la tabla hacia atrás) y las segundas como el cambio de monedas con monedas impares, y comprobar la igualdad hasta $n = 300$. Comprobar también la biyección de Glaisher: en una partición en partes impares, si la parte $k$ aparece $m$ veces, se escribe $m$ en binario y se sustituye por las partes $k \cdot 2^j$ de los bits $j$ que valen 1.

In [4]:
def partes_distintas(n):
    D = [1] + [0] * n
    for parte in range(1, n + 1):
        for m in range(n, parte - 1, -1):
            D[m] += D[m - parte]
    return D[n]


def partes_impares(n):
    I = [1] + [0] * n
    for parte in range(1, n + 1, 2):
        for m in range(parte, n + 1):
            I[m] += I[m - parte]
    return I[n]


def glaisher(impar):
    distintas = []
    for k, m in Counter(impar).items():
        distintas += [k << j for j in range(m.bit_length()) if m >> j & 1]
    return tuple(sorted(distintas, reverse=True))


for p in (p for p in genera_particiones(8) if all(x % 2 for x in p)):
    print(f"{' + '.join(map(str, p)):<30} → {' + '.join(map(str, glaisher(p)))}")

for n in range(301):
    assert partes_distintas(n) == partes_impares(n)
for n in range(1, 31):
    impares = [p for p in genera_particiones(n) if all(x % 2 for x in p)]
    distintas = {p for p in genera_particiones(n) if len(set(p)) == len(p)}
    imagenes = [glaisher(p) for p in impares]
    assert len(set(imagenes)) == len(imagenes) and set(imagenes) == distintas
print(f"\nCon n = 300 hay {partes_distintas(300):,} de cada tipo")

7 + 1                          → 7 + 1
5 + 3                          → 5 + 3
5 + 1 + 1 + 1                  → 5 + 2 + 1
3 + 3 + 1 + 1                  → 6 + 2
3 + 1 + 1 + 1 + 1 + 1          → 4 + 3 + 1
1 + 1 + 1 + 1 + 1 + 1 + 1 + 1  → 8



Con n = 300 hay 114,872,472,064 de cada tipo


## 5. Recorridos de k pasos en una rejilla con obstáculos

En *Caminos en una rejilla* el robot solo va a la derecha o hacia abajo. Ahora un dron de reparto se mueve por un barrio en las cuatro direcciones, una casilla por minuto, sin salir de la rejilla ni entrar en los edificios (`#`), y puede pasar varias veces por la misma casilla. Contar los recorridos de exactamente $k$ minutos que van del almacén `A` al cliente `C`: si $R_t(x)$ es el número de formas de estar en la casilla $x$ tras $t$ minutos, $R_t(x) = \sum_{y \text{ vecina de } x} R_{t-1}(y)$, en $O(k \cdot f \cdot c)$. Escribirlo de arriba abajo y de abajo arriba, compararlo contra la fuerza bruta de las $4^k$ secuencias de movimientos y comprobar que es 0 cuando $k$ no tiene la misma paridad que la distancia Manhattan (cada paso cambia el color de la casilla, como en un tablero de ajedrez).

In [5]:
DIRECCIONES = [(-1, 0), (1, 0), (0, -1), (0, 1)]


def buscar(rejilla, letra):
    return next((f, c) for f, fila in enumerate(rejilla) for c, x in enumerate(fila) if x == letra)


def libre(rejilla, f, c):
    return 0 <= f < len(rejilla) and 0 <= c < len(rejilla[0]) and rejilla[f][c] != "#"


def recorridos_memo(rejilla, k):
    destino = buscar(rejilla, "C")

    @lru_cache(maxsize=None)
    def R(f, c, t):
        if t == 0:
            return int((f, c) == destino)
        return sum(R(f + df, c + dc, t - 1) for df, dc in DIRECCIONES if libre(rejilla, f + df, c + dc))

    return R(*buscar(rejilla, "A"), k)


def recorridos(rejilla, k):
    filas, columnas = len(rejilla), len(rejilla[0])
    R = [[0] * columnas for _ in range(filas)]
    fa, ca = buscar(rejilla, "A")
    R[fa][ca] = 1
    for _ in range(k):
        R = [[sum(R[f + df][c + dc] for df, dc in DIRECCIONES if libre(rejilla, f + df, c + dc))
              if libre(rejilla, f, c) else 0 for c in range(columnas)] for f in range(filas)]
    fc, cc = buscar(rejilla, "C")
    return R[fc][cc]


def recorridos_fuerza_bruta(rejilla, k):
    origen, destino, total = buscar(rejilla, "A"), buscar(rejilla, "C"), 0
    for movimientos in itertools.product(DIRECCIONES, repeat=k):
        f, c = origen
        for df, dc in movimientos:
            f, c = f + df, c + dc
            if not libre(rejilla, f, c):
                break
        else:
            total += (f, c) == destino
    return total


barrio = ["A..#.",
          ".#...",
          "...#C",
          ".#..."]
distancia = sum(abs(a - b) for a, b in zip(buscar(barrio, "A"), buscar(barrio, "C")))
print(f"Distancia Manhattan de A a C: {distancia}")
for k in range(4, 15):
    r = recorridos(barrio, k)
    assert r == recorridos_memo(barrio, k)
    assert (r == 0) or (k - distancia) % 2 == 0
    print(f"k = {k:>2}: {r:>6} recorridos")
for k in range(9):
    assert recorridos(barrio, k) == recorridos_fuerza_bruta(barrio, k)
print(f"k = 60: {recorridos(barrio, 60):,}")

Distancia Manhattan de A a C: 6
k =  4:      0 recorridos
k =  5:      0 recorridos
k =  6:      1 recorridos
k =  7:      0 recorridos
k =  8:     13 recorridos
k =  9:      0 recorridos
k = 10:    108 recorridos
k = 11:      0 recorridos
k = 12:    745 recorridos
k = 13:      0 recorridos
k = 14:   4665 recorridos
k = 60: 325,410,820,012,136,333,768


## 6. Sumas con n dados

Calcular la distribución exacta de la suma de $n$ dados de seis caras: si $D_n(s)$ es el número de tiradas de $n$ dados que suman $s$, el último dado saca de 1 a 6 y $D_n(s) = \sum_{c=1}^{6} D_{n-1}(s - c)$, con $D_0(0) = 1$; cuesta $O(n^2)$ frente a las $6^n$ tiradas. Escribirlo de arriba abajo y de abajo arriba, comprobarlo contra `itertools.product` con pocos dados, comprobar la simetría $D_n(s) = D_n(7n - s)$ y comparar las probabilidades exactas (como `Fraction`) de 5 dados con una simulación de 100 000 tiradas con semilla fija.

In [6]:
def formas_suma_memo(n, s, caras=6):
    @lru_cache(maxsize=None)
    def D(dados, suma):
        if dados == 0:
            return int(suma == 0)
        return sum(D(dados - 1, suma - c) for c in range(1, caras + 1) if c <= suma)

    return D(n, s)


def distribucion(n, caras=6):
    D = {0: 1}
    for _ in range(n):
        nueva = Counter()
        for suma, formas in D.items():
            for c in range(1, caras + 1):
                nueva[suma + c] += formas
        D = dict(nueva)
    return D


for n in range(1, 5):
    D = distribucion(n)
    assert D == dict(Counter(map(sum, itertools.product(range(1, 7), repeat=n))))
    assert all(D[s] == formas_suma_memo(n, s) == D[7 * n - s] for s in D)
    assert sum(D.values()) == 6 ** n

n, tiradas = 5, 100_000
rng = random.Random(2026)
simulacion = Counter(sum(rng.randint(1, 6) for _ in range(n)) for _ in range(tiradas))
exacta = {s: Fraction(f, 6 ** n) for s, f in distribucion(n).items()}
print(f"{'suma':>4}{'formas':>8}{'exacta':>10}{'simulada':>10}")
for s in range(n, 6 * n + 1, 2):
    print(f"{s:>4}{distribucion(n)[s]:>8}{float(exacta[s]):>10.4f}{simulacion[s] / tiradas:>10.4f}")
assert max(abs(float(exacta[s]) - simulacion[s] / tiradas) for s in exacta) < 0.005
assert sum(s * p for s, p in exacta.items()) == Fraction(7 * n, 2)
print(f"\nCon 10 dados, P(suma = 35) = {Fraction(distribucion(10)[35], 6 ** 10)}")

suma  formas    exacta  simulada
   5       1    0.0001    0.0001
   7      15    0.0019    0.0019
   9      70    0.0090    0.0091
  11     205    0.0264    0.0263
  13     420    0.0540    0.0552
  15     651    0.0837    0.0839
  17     780    0.1003    0.0992
  19     735    0.0945    0.0934
  21     540    0.0694    0.0696
  23     305    0.0392    0.0395
  25     126    0.0162    0.0158
  27      35    0.0045    0.0050
  29       5    0.0006    0.0006

Con 10 dados, P(suma = 35) = 7631/104976


## 7. El ladrón de casas

En una calle hay casas con cierto botín, pero si se entra en dos casas vecinas salta la alarma. Calcular el máximo botín sin entrar en dos casas contiguas: la casa $i$ se deja ($B_{i-1}$) o se roba y la anterior se deja ($B_{i-2} + v_i$), así que $B_i = \max(B_{i-1},\ B_{i-2} + v_i)$, en $O(n)$ con dos variables. Escribir la versión memoizada, la de dos variables y una con tabla que diga qué casas elegir. Resolver también la variante en la que las casas forman un círculo (la primera y la última son vecinas): o se deja la primera o se deja la última. Comprobarlo todo contra la fuerza bruta con máscaras de bits.

In [7]:
def botin_memo(valores):
    @lru_cache(maxsize=None)
    def B(i):
        return 0 if i < 0 else max(B(i - 1), B(i - 2) + valores[i])

    return B(len(valores) - 1)


def botin(valores):
    anterior, actual = 0, 0
    for v in valores:
        anterior, actual = actual, max(actual, anterior + v)
    return actual


def casas_elegidas(valores):
    B = [0, 0]
    for v in valores:
        B.append(max(B[-1], B[-2] + v))
    elegidas, i = [], len(valores)
    while i > 0:
        if B[i + 1] == B[i]:
            i -= 1
        else:
            elegidas.append(i - 1)
            i -= 2
    return elegidas[::-1]


def botin_circular(valores):
    if len(valores) == 1:
        return valores[0]
    return max(botin(valores[:-1]), botin(valores[1:]))


def botin_fuerza_bruta(valores, circular=False):
    n, mejor = len(valores), 0
    for mascara in range(1 << n):
        if mascara & (mascara >> 1) or (circular and n > 1 and mascara & 1 and mascara >> (n - 1) & 1):
            continue
        mejor = max(mejor, sum(v for i, v in enumerate(valores) if mascara >> i & 1))
    return mejor


calle = [12, 7, 1, 30, 8, 2, 4, 15, 9, 11]
elegidas = casas_elegidas(calle)
print(f"Botín máximo {botin(calle)} entrando en las casas {elegidas}; en círculo: {botin_circular(calle)}")
assert sum(calle[i] for i in elegidas) == botin(calle) == botin_memo(calle)

random.seed(5)
for _ in range(500):
    valores = [random.randint(0, 20) for _ in range(random.randint(1, 13))]
    elegidas = casas_elegidas(valores)
    assert all(b - a >= 2 for a, b in itertools.pairwise(elegidas))
    assert botin(valores) == botin_memo(valores) == sum(valores[i] for i in elegidas) == botin_fuerza_bruta(valores)
    assert botin_circular(valores) == botin_fuerza_bruta(valores, circular=True)

Botín máximo 70 entrando en las casas [0, 3, 5, 7, 9]; en círculo: 65


## 8. Descodificaciones: listarlas y la k-ésima

En *Problemas de entrevista* se cuentan las formas de descodificar una cadena de cifras con A = 1, B = 2, …, Z = 26 (`"226"` es BBF, BZ o VF). Ahora, con $D_i$ el número de descodificaciones del sufijo que empieza en la posición $i$ (de abajo arriba, desde el final), listar todas las descodificaciones en orden alfabético sin entrar en ramas sin salida (las que tienen $D = 0$, como las que dejan un `0` suelto) y obtener directamente la $k$-ésima sin generar las anteriores: si la primera letra se toma con una cifra hay $D_{i+1}$ descodificaciones que empiezan así; si $k$ es mayor, se salta ese bloque. Comprobarlo con la versión memoizada y contra la fuerza bruta.

In [8]:
def letra(cifras):
    valor = int(cifras)
    return chr(ord("A") + valor - 1) if cifras[0] != "0" and 1 <= valor <= 26 else None


def tabla_descodificaciones(texto):
    D = [0] * len(texto) + [1]
    for i in range(len(texto) - 1, -1, -1):
        D[i] = (D[i + 1] if letra(texto[i]) else 0) + (D[i + 2] if i + 2 <= len(texto) and letra(texto[i:i + 2]) else 0)
    return D


def cuenta_memo(texto):
    @lru_cache(maxsize=None)
    def D(i):
        if i == len(texto):
            return 1
        return sum(D(i + t) for t in (1, 2) if i + t <= len(texto) and letra(texto[i:i + t]))

    return D(0)


def descodificaciones(texto):
    D = tabla_descodificaciones(texto)

    def desde(i):
        if i == len(texto):
            yield ""
        for t in (1, 2):
            if i + t <= len(texto) and letra(texto[i:i + t]) and D[i + t]:
                for resto in desde(i + t):
                    yield letra(texto[i:i + t]) + resto

    return list(desde(0))


def k_esima(texto, k):
    """Descodificación número k (desde 1) en orden alfabético."""
    D = tabla_descodificaciones(texto)
    if not 1 <= k <= D[0]:
        raise IndexError(f"solo hay {D[0]} descodificaciones")
    resultado, i = [], 0
    while i < len(texto):
        if letra(texto[i]) and k <= D[i + 1]:
            t = 1
        else:
            k -= D[i + 1] if letra(texto[i]) else 0
            t = 2
        resultado.append(letra(texto[i:i + t]))
        i += t
    return "".join(resultado)


def descodificaciones_fuerza_bruta(texto):
    resultado = set()
    for cortes in itertools.product((1, 2), repeat=len(texto)):
        trozos, i = [], 0
        for t in cortes:
            if i >= len(texto):
                break
            trozos.append(texto[i:i + t])
            i += t
        if i == len(texto) and all(map(letra, trozos)):
            resultado.add("".join(map(letra, trozos)))
    return sorted(resultado)


for texto in ["226", "1126", "2101", "100", "27"]:
    print(f"{texto:>5}: {descodificaciones(texto)}")
assert k_esima("1126", 3) == "ALF"

random.seed(8)
for _ in range(300):
    texto = "".join(random.choice("0112226789") for _ in range(random.randint(1, 10)))
    lista = descodificaciones(texto)
    assert lista == descodificaciones_fuerza_bruta(texto)
    assert len(lista) == cuenta_memo(texto) == tabla_descodificaciones(texto)[0]
    assert [k_esima(texto, k) for k in range(1, len(lista) + 1)] == lista

largo = "1" * 60
total = tabla_descodificaciones(largo)[0]
print(f"\n'1' × 60 tiene {total:,} descodificaciones; la número 10^12 es {k_esima(largo, 10**12)}")
try:
    k_esima("100", 1)
except IndexError as e:
    print("'100':", e)

  226: ['BBF', 'BZ', 'VF']
 1126: ['AABF', 'AAZ', 'ALF', 'KBF', 'KZ']
 2101: ['BJA']
  100: []
   27: ['BG']



'1' × 60 tiene 2,504,730,781,961 descodificaciones; la número 10^12 es AKAAAAAKAKAKKAAAAAAAKAAAKAAKKAAAAKKAKKAAKAAKA
'100': solo hay 0 descodificaciones


## 9. Partir una cadena en palabras

Un texto ha perdido los espacios. Dado un diccionario, decidir si se puede partir en palabras, contar de cuántas formas y listarlas. Si $S_i$ dice si el sufijo que empieza en $i$ se puede partir, $S_n$ es cierto y $S_i$ lo es si alguna palabra del diccionario empieza en $i$ y termina en una posición $j$ con $S_j$ cierto; mirando solo palabras de hasta $L$ letras cuesta $O(n \cdot L)$. Escribir el recuento de abajo arriba y la lista de particiones de arriba abajo con `lru_cache`, y comprobarlo contra la fuerza bruta que prueba los $2^{n-1}$ juegos de cortes.

In [9]:
diccionario = {"la", "isla", "is", "bonita", "boni", "ta", "de", "del", "el", "mar", "a", "ar", "mara"}
L = max(map(len, diccionario))


def cuenta_particiones(texto):
    N = [0] * len(texto) + [1]
    for i in range(len(texto) - 1, -1, -1):
        N[i] = sum(N[j] for j in range(i + 1, min(len(texto), i + L) + 1) if texto[i:j] in diccionario)
    return N[0]


def particiones_en_palabras(texto):
    @lru_cache(maxsize=None)
    def desde(i):
        if i == len(texto):
            return [[]]
        return [[texto[i:j], *resto] for j in range(i + 1, min(len(texto), i + L) + 1)
                if texto[i:j] in diccionario for resto in desde(j)]

    return [" ".join(p) for p in desde(0)]


def particiones_fuerza_bruta(texto):
    resultado = []
    for mascara in range(1 << (len(texto) - 1)):
        cortes = [0] + [i + 1 for i in range(len(texto) - 1) if mascara >> i & 1] + [len(texto)]
        palabras = [texto[a:b] for a, b in itertools.pairwise(cortes)]
        if all(p in diccionario for p in palabras):
            resultado.append(" ".join(palabras))
    return sorted(resultado)


for texto in ["laislabonitadelmar", "laislabonitadelmarx", "mardelaisla"]:
    print(f"{texto} ({cuenta_particiones(texto)}):", *particiones_en_palabras(texto), sep="\n  ")
    assert sorted(particiones_en_palabras(texto)) == particiones_fuerza_bruta(texto)
    assert cuenta_particiones(texto) == len(particiones_en_palabras(texto))

random.seed(9)
palabras = sorted(diccionario)
for _ in range(200):
    texto = "".join(random.choice(palabras) for _ in range(random.randint(1, 4)))[:14]
    if random.random() < 0.3:
        texto += random.choice("xr")
    assert cuenta_particiones(texto) == len(particiones_fuerza_bruta(texto))
largo = "laisla" * 25
print(f"\n'laisla' × 25 ({len(largo)} letras) se puede partir de {cuenta_particiones(largo):,} formas")
assert cuenta_particiones(largo) == 2 ** 25

laislabonitadelmar (4):
  la is la boni ta del mar
  la is la bonita del mar
  la isla boni ta del mar
  la isla bonita del mar


laislabonitadelmarx (0):


mardelaisla (4):
  mar de la is la
  mar de la isla
  mar del a is la
  mar del a isla



'laisla' × 25 (150 letras) se puede partir de 33,554,432 formas


## 10. La palabra más larga escondida en una ristra de letras

Dada una ristra de letras (por ejemplo, las fichas que han salido en un juego) y un diccionario, encontrar la palabra más larga que se puede formar como subsecuencia, es decir, tachando letras sin cambiar el orden de las que quedan (a igual longitud, la primera en orden alfabético). Precalcular de abajo arriba la tabla $\text{sig}[i][x]$, la primera posición $\ge i$ donde aparece la letra $x$ ($\text{sig}[i] = \text{sig}[i + 1]$ salvo $\text{sig}[i][s_i] = i$), con la que cada palabra se comprueba en tantos pasos como letras tenga, sin recorrer la ristra. Compararlo con la comprobación con dos punteros y con la fuerza bruta que genera todas las subsecuencias de una ristra corta.

In [10]:
def tabla_siguiente(ristra):
    sig = [{} for _ in range(len(ristra) + 1)]
    for i in range(len(ristra) - 1, -1, -1):
        sig[i] = {**sig[i + 1], ristra[i]: i}
    return sig


def es_subsecuencia_tabla(palabra, sig):
    i = 0
    for x in palabra:
        if x not in sig[i]:
            return False
        i = sig[i][x] + 1
    return True


def es_subsecuencia(palabra, ristra):
    letras = iter(ristra)
    return all(x in letras for x in palabra)


def mas_larga(ristra, palabras):
    sig = tabla_siguiente(ristra)
    validas = [p for p in palabras if es_subsecuencia_tabla(p, sig)]
    return min(validas, key=lambda p: (-len(p), p), default=None)


def mas_larga_fuerza_bruta(ristra, palabras):
    subsecuencias = {"".join(c) for r in range(len(ristra) + 1) for c in itertools.combinations(ristra, r)}
    validas = [p for p in palabras if p in subsecuencias]
    return min(validas, key=lambda p: (-len(p), p), default=None)


vocabulario = ["cardon", "cardonal", "tabaiba", "costa", "casa", "drago", "sabina", "barco", "roca", "arena",
               "acantilado", "lava", "volcan", "bahia", "cala", "playa", "pino", "tasca", "brisa", "nasa"]
ristra = "cvardosnaltabcaiybasenlavcostda"
print(f"En '{ristra}':")
sig = tabla_siguiente(ristra)
for p in vocabulario:
    assert es_subsecuencia_tabla(p, sig) == es_subsecuencia(p, ristra)
print("  caben:", sorted((p for p in vocabulario if es_subsecuencia(p, ristra)), key=lambda p: (-len(p), p)))
print("  la más larga:", mas_larga(ristra, vocabulario))

random.seed(10)
for _ in range(200):
    corta = "".join(random.choice("abcdilnorst") for _ in range(random.randint(0, 12)))
    assert mas_larga(corta, vocabulario) == mas_larga_fuerza_bruta(corta, vocabulario)
    for p in vocabulario:
        assert es_subsecuencia_tabla(p, tabla_siguiente(corta)) == es_subsecuencia(p, corta)

En 'cvardosnaltabcaiybasenlavcostda':
  caben: ['cardonal', 'tabaiba', 'cardon', 'sabina', 'volcan', 'arena', 'costa', 'tasca', 'cala', 'casa', 'lava', 'nasa', 'roca']
  la más larga: cardonal
